# Paper 1 — Notebook 2 of 4
## Terrain Controls on Bias (RQ1.2 ⭐) & Statistical Bias Correction

**Run order:** NB1 → **NB2** → NB3 → NB4. Run NB1 first — this notebook reads its parquet outputs.

**What NB2 does**
1. Reloads NB1's paired tables + terrain
2. Landscape-type stratification of skill — *§4.2* (supports F5)
3. Terrain–bias Spearman correlation matrix with significance (T5) — *step 1.5*
4. Nested / parsimonious bias-model comparison, R² progression (T6)
5. Statistical bias correction: **Linear Scaling, EQM, TQM** for Tmax/Tmin/precip — *step 1.6*
   evaluated under a **temporal split** (train 2000-2015, test 2016-2023)
6. Recomputes rainfall detection **after** statistical correction → fills the 'corrected' rows of T4

**Outputs:** `T5_terrain_correlation.csv`, `T6_nested_bias_model.csv`,
`landscape_stratified_skill.csv`, `stat_correction_metrics.csv`,
`T4_rainfall_detection_full.csv`, `corrected_stat.parquet` (for comparison in NB3)


### Cell 1 — Imports and reload NB1 outputs
Same BASE/OUT resolution logic; then load the parquet working tables.

In [1]:
import os, warnings, numpy as np, pandas as pd
from scipy import stats
warnings.filterwarnings('ignore')
pd.set_option('display.width',170); pd.set_option('display.max_columns',60)

# ----- PATHS -----------------------------------------------------------------
# 1. Base Dataset Path (Weather Dataset)
# 'dataset/' ফোল্ডারটি যুক্ত করা হয়েছে কারণ ডেটাগুলো এর ভেতরে রয়েছে
DS_CANDS = [
    '/kaggle/input/datasets/neloypramanik4444/weather-dataset/dataset/', 
    '/kaggle/input/weather-dataset/dataset/', 
    './'
]
BASE = next((c for c in DS_CANDS if os.path.exists(c)), './')

# 2. Notebook Input Path (WeatherData Paper-V1.1)
NB_CANDS = [
    '/kaggle/input/notebooks/neloypramanik4444/weatherdata-paper-v1-1/',
    '/kaggle/input/weatherdata-paper-v1-1/',
    './out/', 
    '/kaggle/working/'
]
NB_BASE = next((c for c in NB_CANDS if os.path.exists(c)), './out/')

# 3. Output Path
OUT = '/kaggle/working/' if os.path.exists('/kaggle/working/') else './out/'
os.makedirs(OUT, exist_ok=True)

print(f'BASE = {BASE} \nNB_BASE = {NB_BASE} \nOUT = {OUT}\n')

# ----- LOAD DATA -------------------------------------------------------------
# এখন parquet এবং meta ফাইলগুলো NB_BASE থেকে রিড করবে
def rd(name): return pd.read_parquet(os.path.join(NB_BASE, name))

paired_era5 = rd('paired_era5.parquet')
meta = pd.read_csv(os.path.join(NB_BASE, 'meta_with_cells.csv'))

# terrain ফাইলগুলো BASE এর নির্দিষ্ট সাব-ফোল্ডার থেকে রিড করবে
terrain_dir = os.path.join(BASE, 'C6_Terrain (slope, TPI, TWI)')
ter = pd.read_csv(os.path.join(terrain_dir, 'terrain_full.csv'))
skill = pd.read_csv(os.path.join(terrain_dir, 'terrain_skill_merged_v2.csv'))

print('paired_era5:', paired_era5.shape, '| terrain:', ter.shape, '| skill:', skill.shape)

# ----- METRIC HELPERS --------------------------------------------------------
# metric helpers (repeated so NB2 is standalone)
def kge(sim,obs):
    sim=np.asarray(sim,float);obs=np.asarray(obs,float);m=np.isfinite(sim)&np.isfinite(obs)
    sim,obs=sim[m],obs[m]
    if len(obs)<3:return np.nan
    r=np.corrcoef(sim,obs)[0,1];a=sim.std()/obs.std() if obs.std() else np.nan;b=sim.mean()/obs.mean() if obs.mean() else np.nan
    return 1-np.sqrt((r-1)**2+(a-1)**2+(b-1)**2)

def metrics_block(sim,obs):
    sim=np.asarray(sim,float);obs=np.asarray(obs,float);m=np.isfinite(sim)&np.isfinite(obs)
    sim,obs=sim[m],obs[m]
    if len(obs)<3:return dict(n=len(obs),r=np.nan,bias=np.nan,mae=np.nan,rmse=np.nan,kge=np.nan)
    return dict(n=len(obs),r=np.corrcoef(sim,obs)[0,1],bias=(sim-obs).mean(),
                mae=np.abs(sim-obs).mean(),rmse=np.sqrt(((sim-obs)**2).mean()),kge=kge(sim,obs))

def contingency(sim,obs,thr):
    sim=np.asarray(sim,float);obs=np.asarray(obs,float);m=np.isfinite(sim)&np.isfinite(obs)
    sim,obs=sim[m],obs[m];fo=obs>=thr;fs=sim>=thr
    H=int((fo&fs).sum());M=int((fo&~fs).sum());F=int((~fo&fs).sum());C=int((~fo&~fs).sum());n=H+M+F+C
    pod=H/(H+M) if(H+M)else np.nan;far=F/(H+F) if(H+F)else np.nan;csi=H/(H+M+F) if(H+M+F)else np.nan
    if n: acc=(H+C)/n;rand=((H+M)*(H+F)+(C+M)*(C+F))/n**2;hss=(acc-rand)/(1-rand) if(1-rand)else np.nan
    else: hss=np.nan
    return dict(threshold=thr,H=H,M=M,F=F,C=C,POD=pod,FAR=far,CSI=csi,HSS=hss)

RAIN_THRESHOLDS=[1,10,20,50]; FOCAL_FILLED=['Kutubdia','Sandwip']
print('helpers ready')

BASE = /kaggle/input/datasets/neloypramanik4444/weather-dataset/dataset/ 
NB_BASE = /kaggle/input/notebooks/neloypramanik4444/weatherdata-paper-v1-1/ 
OUT = /kaggle/working/

paired_era5: (314326, 29) | terrain: (36, 16) | skill: (36, 21)
helpers ready


### Cell 2 — Landscape-type stratification of skill (§4.2)
We label each station Interior / Coastal-island / Hill-adjacent from `land_frac_9km` and `elev_std_9km`,
then summarise Tmax skill by group. This reproduces the coastal cold-bias amplification (−2.6 to −2.9 °C).

In [2]:
def landscape(row):
    if row['elev_std_9km']>=12:            # strong sub-grid relief
        return 'Hill-adjacent'
    if row['land_frac_9km']<0.80:          # cell mixes substantial water
        return 'Coastal / island'
    return 'Interior plain'
skill=skill.copy()
skill['landscape']=skill.apply(landscape, axis=1)

strat=(skill.groupby('landscape')
       .agg(n=('station_name','size'),
            tmax_r=('tmax_r','mean'),
            tmax_bias=('tmax_bias','mean'),
            tmin_r=('tmin_r','mean'),
            prcp_r=('prcp_r','mean'),
            land_frac=('land_frac_9km','mean'),
            elev_std=('elev_std_9km','mean'))
       .round(3).reset_index())
skill[['station_name','landscape']].to_csv(OUT+'station_landscape.csv', index=False)
strat.to_csv(OUT+'landscape_stratified_skill.csv', index=False)
print('=== Landscape-stratified skill (§4.2) ===')
print(strat.to_string(index=False))
print('\nExample members:')
for lt in strat['landscape']:
    print(' ', lt, ':', ', '.join(skill.loc[skill.landscape==lt,'station_name'].head(6)))

=== Landscape-stratified skill (§4.2) ===
       landscape  n  tmax_r  tmax_bias  tmin_r  prcp_r  land_frac  elev_std
Coastal / island  5   0.899     -2.303   0.957   0.514      0.405     3.558
   Hill-adjacent  6   0.879     -2.158   0.962   0.464      0.885    28.811
  Interior plain 25   0.911     -1.374   0.966   0.397      0.967     3.334

Example members:
  Coastal / island : Cox's Bazar, Hatiya, Kutubdia, Mongla, Sandwip
  Hill-adjacent : Ambagan (Ctg), Rangamati, Sitakunda, Srimangal, Sylhet, Teknaf
  Interior plain : Barisal, Bhola, Bogra, Chandpur, Chittagong, Chuadanga


### Cell 3 — Terrain–bias Spearman matrix with significance (T5) — *step 1.5*
Spearman ρ (n=36) between each terrain variable and four skill metrics, with p-values.
Reproduces §4.3: land_frac drives cold bias (+0.51**), relief degrades correlation, water proximity
controls Tmin/precip skill.

In [3]:
terr_vars=['land_frac_9km','elev_std_9km','roughness_5km','dist_water_km',
           'elev_dem','northness','eastness','tpi_5km','slope_mean_9km']
skill_vars=['tmax_bias','tmax_r','tmin_r','prcp_r']
rows=[]
for tv in terr_vars:
    rec={'terrain_var':tv}
    for sv in skill_vars:
        rho,p=stats.spearmanr(skill[tv], skill[sv])
        star='**' if p<0.01 else ('*' if p<0.05 else '')
        rec[sv]=f'{rho:+.2f}{star}'
        rec[sv+'_p']=round(p,4)
    rows.append(rec)
T5=pd.DataFrame(rows)
disp_cols=['terrain_var']+skill_vars
T5.to_csv(OUT+'T5_terrain_correlation.csv', index=False)
print('=== TABLE T5 — Terrain vs skill (Spearman rho, n=36) ===')
print('  * p<0.05   ** p<0.01')
print(T5[disp_cols].to_string(index=False))

=== TABLE T5 — Terrain vs skill (Spearman rho, n=36) ===
  * p<0.05   ** p<0.01
   terrain_var tmax_bias tmax_r  tmin_r  prcp_r
 land_frac_9km   +0.51**  +0.13 +0.46** -0.51**
  elev_std_9km     -0.31 -0.38*   -0.25   +0.22
 roughness_5km     -0.31 -0.39*   -0.26   +0.23
 dist_water_km     +0.26  +0.32 +0.65** -0.61**
      elev_dem     +0.31  +0.10 +0.58** -0.52**
     northness    -0.35*  +0.00  -0.36*   +0.27
      eastness     +0.21  -0.24   -0.09   -0.01
       tpi_5km     +0.07  +0.33   +0.18   -0.24
slope_mean_9km     -0.09  -0.31   -0.12   +0.09


### Cell 4 — Nested / parsimonious bias-model comparison (T6)
OLS on Tmax bias with growing predictor sets, reporting R² and adjusted R².
Reproduces §4.4: land_frac (0.30) → +elev_std (0.43, the sweet spot) → slope adds only ~+0.02
(**reported as a negative result** — slope is not useful on a flat delta; with n=36, >3 predictors overfit).

In [4]:
from itertools import accumulate
import statsmodels.api as sm
predictor_sets=[
    ['land_frac_9km'],
    ['land_frac_9km','elev_std_9km'],
    ['land_frac_9km','elev_std_9km','slope_mean_9km'],
    ['land_frac_9km','elev_std_9km','elev_dem','dist_water_km'],
]
y=skill['tmax_bias'].values
rows=[]
for ps in predictor_sets:
    X=sm.add_constant(skill[ps].values)
    m=sm.OLS(y,X).fit()
    rows.append(dict(predictors=' + '.join(ps), k=len(ps),
                     R2=round(m.rsquared,3), adjR2=round(m.rsquared_adj,3),
                     AIC=round(m.aic,1)))
T6=pd.DataFrame(rows)
T6.to_csv(OUT+'T6_nested_bias_model.csv', index=False)
print('=== TABLE T6 — Nested bias-model comparison (Tmax bias) ===')
print(T6.to_string(index=False))
delta=T6.loc[2,'R2']-T6.loc[1,'R2']
print(f'\nSlope contribution beyond land_frac+elev_std: +{delta:.3f} R2  -> negative result: slope not useful on flat delta.')
print('Chosen parsimonious model: land_frac_9km + elev_std_9km  (R2 = %.3f)'%T6.loc[1,'R2'])

=== TABLE T6 — Nested bias-model comparison (Tmax bias) ===
                                             predictors  k    R2  adjR2  AIC
                                          land_frac_9km  1 0.303  0.283 51.3
                           land_frac_9km + elev_std_9km  2 0.428  0.394 46.1
          land_frac_9km + elev_std_9km + slope_mean_9km  3 0.447  0.395 46.9
land_frac_9km + elev_std_9km + elev_dem + dist_water_km  4 0.430  0.356 50.0

Slope contribution beyond land_frac+elev_std: +0.019 R2  -> negative result: slope not useful on flat delta.
Chosen parsimonious model: land_frac_9km + elev_std_9km  (R2 = 0.428)


### Cell 5 — Statistical bias correction methods — *step 1.6*
Three classical corrections, fit **per station** on a temporal split (train 2000-2015 → test 2016-2023):

* **Linear Scaling (LS)** — additive shift for temperature, multiplicative for precip (mean-matching)
* **Empirical Quantile Mapping (EQM)** — map the model CDF onto the obs CDF via empirical quantiles
* **Tricub/Transfer Quantile Mapping (TQM)** — smooth quantile transfer with tail extrapolation

These are the non-ML baselines that the ML models in NB3 must beat.

In [5]:
TRAIN_END='2015-12-31'; TEST_START='2016-01-01'

def fit_ls(mod_tr, obs_tr, kind):
    if kind=='temp':  return ('add', np.nanmean(obs_tr)-np.nanmean(mod_tr))
    ratio=np.nansum(obs_tr)/np.nansum(mod_tr) if np.nansum(mod_tr)>0 else 1.0
    return ('mul', ratio)
def apply_ls(mod, par):
    return mod+par[1] if par[0]=='add' else mod*par[1]

def fit_eqm(mod_tr, obs_tr, nq=100):
    q=np.linspace(0.01,0.99,nq)
    mask=np.isfinite(mod_tr)&np.isfinite(obs_tr)
    mq=np.quantile(mod_tr[mask], q); oq=np.quantile(obs_tr[mask], q)
    return (q,mq,oq)
def apply_eqm(mod, par):
    q,mq,oq=par
    return np.interp(mod, mq, oq, left=oq[0]+(mod.min()-mq[0]) if False else oq[0],
                     right=oq[-1]+(0)).astype(float) if False else np.interp(np.clip(mod,mq[0],mq[-1]),mq,oq)

def fit_tqm(mod_tr, obs_tr, nq=100):
    # transfer function on quantiles + linear tail extrapolation (delta beyond edges)
    q=np.linspace(0.01,0.99,nq)
    mask=np.isfinite(mod_tr)&np.isfinite(obs_tr)
    mq=np.quantile(mod_tr[mask],q); oq=np.quantile(obs_tr[mask],q)
    lo_delta=oq[0]-mq[0]; hi_delta=oq[-1]-mq[-1]
    return (mq,oq,lo_delta,hi_delta)
def apply_tqm(mod, par):
    mq,oq,lo,hi=par
    out=np.interp(mod,mq,oq)
    out=np.where(mod<mq[0], mod+lo, out)
    out=np.where(mod>mq[-1], mod+hi, out)
    return out

METHODS={'LS':(fit_ls,apply_ls),'EQM':(fit_eqm,apply_eqm),'TQM':(fit_tqm,apply_tqm)}
VARS=[('t2m_max','obs_tmax','temp','Tmax'),
      ('t2m_min','obs_tmin','temp','Tmin'),
      ('era5_prcp','obs_prcp','precip','Precip')]

df=paired_era5.copy()
df['is_train']=df['date']<=TRAIN_END
records=[]; corrected_store={}
for mod_c,obs_c,kind,label in VARS:
    for mname,(fitf,appf) in METHODS.items():
        pred_test=np.full(len(df), np.nan)
        for s,idx in df.groupby('station_name').groups.items():
            g=df.loc[idx]
            tr=g[g.is_train]; te_mask=~g.is_train
            mtr=tr[mod_c].values; otr=tr[obs_c].values
            if np.isfinite(mtr).sum()<50 or np.isfinite(otr).sum()<50: continue
            if mname=='LS': par=fitf(mtr,otr,kind)
            else:           par=fitf(mtr,otr)
            corr=appf(g[mod_c].values, par)
            arr=np.full(len(g),np.nan); arr[te_mask.values]=corr[te_mask.values]
            pred_test[[df.index.get_loc(i) for i in idx]]=arr
        te=df[~df.is_train]
        obs=te[obs_c].values; pr=pred_test[~df.is_train.values]
        mb=metrics_block(pr,obs); mb.update(variable=label, method=mname, stage='corrected')
        records.append(mb)
        corrected_store[(label,mname)]=pred_test
    # raw baseline on the SAME test period for fair comparison
    te=df[~df.is_train]
    mb=metrics_block(te[mod_c].values, te[obs_c].values); mb.update(variable=label, method='RAW', stage='raw')
    records.append(mb)

stat_metrics=pd.DataFrame(records)[['variable','method','stage','n','r','bias','mae','rmse','kge']]
for c in ['r','bias','mae','rmse','kge']: stat_metrics[c]=stat_metrics[c].round(3)
stat_metrics.to_csv(OUT+'stat_correction_metrics.csv', index=False)
print('=== Statistical bias correction — test period 2016-2023 ===')
print(stat_metrics.to_string(index=False))

=== Statistical bias correction — test period 2016-2023 ===
variable method     stage      n     r   bias   mae   rmse   kge
    Tmax     LS corrected 103738 0.924 -0.304 1.080  1.401 0.884
    Tmax    EQM corrected 103738 0.930 -0.295 1.052  1.382 0.927
    Tmax    TQM corrected 103738 0.930 -0.296 1.059  1.394 0.921
    Tmax    RAW       raw 103738 0.916 -1.834 2.034  2.324 0.869
    Tmin     LS corrected 102643 0.963 -0.073 1.117  1.431 0.900
    Tmin    EQM corrected 102643 0.967 -0.006 1.000  1.352 0.962
    Tmin    TQM corrected 102643 0.967 -0.006 1.003  1.356 0.960
    Tmin    RAW       raw 102643 0.952  0.095 1.223  1.609 0.905
  Precip     LS corrected 103780 0.464 -0.119 6.785 16.981 0.392
  Precip    EQM corrected 103780 0.444 -0.395 7.055 18.343 0.426
  Precip    TQM corrected 103780 0.440 -0.202 7.200 19.041 0.437
  Precip    RAW       raw 103780 0.455 -0.525 6.655 16.770 0.340


### Cell 6 — Rainfall detection AFTER statistical correction → complete T4
Take the best statistical precip correction (by test KGE) and recompute POD/FAR/CSI/HSS on the test period,
then stack with the raw baseline so the paper can show before/after at each threshold.

In [6]:
prc=stat_metrics.query("variable=='Precip' and stage=='corrected'").sort_values('kge',ascending=False)
best_precip_method=prc.iloc[0]['method']
print('Best statistical precip correction by KGE:', best_precip_method)

te=df[~df.is_train].reset_index(drop=True)
corr_precip=corrected_store[('Precip',best_precip_method)][~df.is_train.values]
obs_precip=te['obs_prcp'].values

t4_rows=[]
for thr in RAIN_THRESHOLDS:
    c=contingency(te['era5_prcp'].values, obs_precip, thr); c.update(product='ERA5-Land', stage='raw (test)')
    t4_rows.append(c)
    c=contingency(corr_precip, obs_precip, thr); c.update(product=f'ERA5-Land+{best_precip_method}', stage='corrected')
    t4_rows.append(c)
T4b=pd.DataFrame(t4_rows)[['product','stage','threshold','H','M','F','C','POD','FAR','CSI','HSS']]
for c in ['POD','FAR','CSI','HSS']: T4b[c]=T4b[c].round(3)

# merge with NB1 raw full-period table for the paper's T4
# Ekhane OUT er bodole NB_BASE dewa hoyeche
T4_raw_full=pd.read_csv(os.path.join(NB_BASE, 'T4_rainfall_detection.csv')) 
T4_full=pd.concat([T4_raw_full.assign(period='2000-2023 full'),
                   T4b.assign(period='2016-2023 test')], ignore_index=True)
T4_full.to_csv(OUT+'T4_rainfall_detection_full.csv', index=False)
print('=== T4 — detection before/after statistical correction (test period) ===')
print(T4b.to_string(index=False))
print('\nDoes correction help heavy rain? Compare POD at 50 mm raw vs corrected above.')

Best statistical precip correction by KGE: TQM
=== T4 — detection before/after statistical correction (test period) ===
      product      stage  threshold     H     M     F     C   POD   FAR   CSI   HSS
    ERA5-Land raw (test)          1 29470  3262 19657 51391 0.900 0.400 0.563 0.549
ERA5-Land+TQM  corrected          1 21423 11309  9096 61952 0.654 0.298 0.512 0.536
    ERA5-Land raw (test)         10  9418  7370 10247 76745 0.561 0.521 0.348 0.415
ERA5-Land+TQM  corrected         10  8155  8633  7673 79319 0.486 0.485 0.333 0.407
    ERA5-Land raw (test)         20  3757  6720  4783 88520 0.359 0.560 0.246 0.335
ERA5-Land+TQM  corrected         20  4270  6207  5965 87338 0.408 0.583 0.260 0.347
    ERA5-Land raw (test)         50   419  3001   854 99506 0.123 0.671 0.098 0.164
ERA5-Land+TQM  corrected         50   980  2440  2526 97834 0.287 0.720 0.165 0.258

Does correction help heavy rain? Compare POD at 50 mm raw vs corrected above.


### Cell 7 — Persist statistically-corrected test predictions for NB3 comparison

In [7]:
keep=df[['station_name','date','is_train','obs_tmax','obs_tmin','obs_prcp',
         't2m_max','t2m_min','era5_prcp']].copy()
for (label,mname),arr in corrected_store.items():
    keep[f'{label}_{mname}']=arr
keep.to_parquet(OUT+'corrected_stat.parquet', index=False)
print('Wrote corrected_stat.parquet with columns:')
print([c for c in keep.columns if any(k in c for k in ['LS','EQM','TQM'])])
print('\nNB2 complete. Next: run Notebook 3 (ML bias correction + LOSO + SHAP).')

Wrote corrected_stat.parquet with columns:
['Tmax_LS', 'Tmax_EQM', 'Tmax_TQM', 'Tmin_LS', 'Tmin_EQM', 'Tmin_TQM', 'Precip_LS', 'Precip_EQM', 'Precip_TQM']

NB2 complete. Next: run Notebook 3 (ML bias correction + LOSO + SHAP).
